In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from langgraph.graph import StateGraph, START
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

In [4]:
load_dotenv()

False

In [ ]:
llm = ChatOpenAI()

In [6]:
client = MultiServerMCPClient(
    {
        "arith": {
            "transport": "stdio",
            "command": "python3",          
            "args": ["/Users/Dhruv/Desktop/mcp-math-server/main.py"],
        },
        "expense": {
            "transport": "streamable_http",  # if this fails, try "sse"
            "url": "https://splendid-gold-dingo.fastmcp.app/mcp"
        }
    }
)

In [7]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [9]:
async def build_graph():
    tools = await client.get_tools()

    print(tools)

    llm_with_tools = llm.bind_tools(tools)

    async def chat_node(stats: ChatState):

        messages = state["messages"]
        response = await llm_with_tools.ainvoke(messages)
        return {'messages' : {response}}

    tool_node = ToolNode(tools)

    graph = StateGraph(ChatState)
    graph.add_node("chat_node", chat_node)
    graph.add_node("tools", tool_node)

    graph.add_edge(START, "chat_node")
    graph.add_conditional_edges("chat_node", tools_condition)
    graph.add_edge("tools", "chat_node")

    chatbot = graph.compile()
    return chatbot
    

In [ ]:
async def main():
    chatbot = await build_graph()

    result = await chatbot.ainvoke({'messages': [HumanMessage(content="Give me all my expenses for the month of Nov from 1 Nov to 30 Nov")]}))

    print(result['messages'][-1].content)

if __name__ == 'main':
    asyncio.run(main())